<a href="https://colab.research.google.com/github/steezym/image-captioning/blob/anggota1-model/anggota1_uji_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Uji Model - Anggota 1 (Ketua)

**Mazmur Bryan**: EncoderCNN, LSTMCell manual, DecoderLSTM, CaptioningModel.

Semua modul dibangun from scratch (tanpa model pretrained, tanpa `nn.LSTM`) dan diuji dengan tensor acak, karena dataset belum dibutuhkan untuk memastikan arsitektur benar.

Cara menjalankan: **Runtime → Run all** (runtime T4 GPU).

## Setup

In [ ]:
# Sambungkan Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Ambil kode dari GitHub
import os
REPO_URL = 'https://github.com/steezym/image-captioning.git'
PATH = '/content/image-captioning'

if not os.path.exists(PATH):
    !git clone -q {REPO_URL} {PATH}
%cd {PATH}

# Pindah ke branch kerja (ganti sesuai peran), lalu ambil versi terbaru
BRANCH = 'main'   # Anggota 1: 'anggota1-model', Anggota 2: 'anggota2-data', dst.
!git fetch -q
!git checkout -q {BRANCH}
!git pull -q origin {BRANCH}

# Muat ulang kode terbaru tanpa restart runtime.
# (%autoreload tidak dipakai karena gagal di Python 3.13: modul 'imp' sudah dihapus)
# Cara pakai setelah git pull:  muat_ulang()  lalu import ulang modulnya
import sys
def muat_ulang():
    for nama in [m for m in sys.modules if m == 'src' or m.startswith('src.')]:
        del sys.modules[nama]
    print('Modul src dimuat ulang pada import berikutnya')

DRIVE_ROOT = '/content/drive/MyDrive/ImageCaptioning'
print('Drive OK:', os.path.exists(DRIVE_ROOT))

Mounted at /content/drive
/content/image-captioning
Drive OK: True


## Cek GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
GPU tersedia: True
GPU: Tesla T4


## 1. EncoderCNN

Vanilla CNN 5 blok (Conv 3×3 → BatchNorm → ReLU → MaxPool) + Global Average Pooling.
Gambar `(B, 3, 128, 128)` → fitur `(B, 256)`.

Yang dicek: shape tiap blok, keluaran `(4, 256)`, dan gradien mengalir sampai lapisan pertama.

In [ ]:
!python -m tests.test_encoder

Shape tiap blok:
  input   : (4, 3, 128, 128)
  blok 1  : (4, 32, 64, 64)
  blok 2  : (4, 64, 32, 32)
  blok 3  : (4, 128, 16, 16)
  blok 4  : (4, 256, 8, 8)
  blok 5  : (4, 256, 4, 4)
  keluaran: (4, 256)

Jumlah parameter: 1,045,024
Device: cuda
LULUS: EncoderCNN (4, 3, 128, 128) -> (4, 256)


## 2. LSTMCell manual

Sel LSTM ditulis sendiri dari `nn.Linear`: 4 gate (input, forget, candidate, output), bias forget gate = 1.

Yang dicek: shape, rumus gate dihitung ulang secara terpisah, bias forget, rentang `h`, dan gradien setelah 30 langkah.

In [ ]:
!python -m tests.test_lstm_cell

Satu langkah : x (4, 256) -> h (4, 512), c (4, 512)
Rumus gate   : sesuai (c' = f*c + i*g, h' = o*tanh(c'))
Bias forget  : 1.0 (gate lain dibiarkan acak)
Rentang h    : [-0.249, 0.305] (harus di dalam -1..1)
30 langkah   : gradien ke kata pertama = 0.1518 (> 0)

Jumlah parameter: 1,574,912
Device: cuda
LULUS: LSTMCell manual


## 3. DecoderLSTM

Embedding → LSTMCell → Dropout → Linear. Fitur gambar menjadi memori awal (`h0`, `c0`). Training memakai teacher forcing, pembuatan caption memakai greedy search.

Yang dicek: shape `(4, 29, vocab)`, tidak mengintip kata masa depan, fitur gambar dipakai, embedding `<pad>` = 0, gradien, dan `generate`.

In [ ]:
!python -m tests.test_decoder

forward      : fitur (4, 256) + caption (4, 30) -> logits (4, 29, 1000)
Teacher force: tebakan langkah t hanya bergantung pada kata 0..t (tidak mengintip)
Fitur gambar : gambar berbeda menghasilkan tebakan berbeda
Embedding pad: vektor <pad> = 0
Gradien      : mengalir sampai fitur gambar
generate     : 29 kata, contoh nomor kata [750, 568, 893, 893, 893, 913, 163, 913]...
               (masih acak karena belum dilatih)

Jumlah parameter (vocab 1000): 2,607,080
Device: cuda
LULUS: DecoderLSTM


## 4. CaptioningModel + uji hafalan

Encoder + decoder digabung dalam satu model. **Uji hafalan**: model dilatih 150 langkah pada 4 pasangan gambar-caption acak. Jika semua bagian tersambung benar, loss turun drastis dan model bisa menulis ulang caption yang dihafal.

In [ ]:
!python -m tests.test_model

forward : gambar (4, 3, 128, 128) + caption (4, 30) -> logits (4, 29, 1000)
caption : 'w156 w789 w547 w772 w599 w253 w253 w751 w288 w288 w825 w82 w...' (masih acak, belum dilatih)
gradien : mengalir dari keluaran decoder sampai Conv pertama encoder

Uji hafalan (loss awal ideal ~ ln(1000) = 6.91):
  langkah   1: loss 6.9234
  langkah  25: loss 0.4399
  langkah  50: loss 0.0454
  langkah 100: loss 0.0122
  langkah 150: loss 0.0066
  waktu: 4.8 detik
  caption yang ditulis ulang dengan benar: 4/4

Jumlah parameter (vocab 1000): 3,652,104
Device: cuda
LULUS: CaptioningModel


## Ringkasan

Jika keempat uji menampilkan `LULUS`, arsitektur model siap dipakai untuk training dengan data Flickr8k (Anggota 2 & 3).